In [1]:
import os
import psycopg2
import pandas as pd
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from openai import OpenAI
import openpyxl

pd.set_option("display.max_colwidth", None)

In [2]:
import boto3
import os 
from IPython.display import display
import json
from typing import Any
from datetime import datetime

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
print("OK")

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OK


In [5]:
prefix = datetime.now().strftime("%d_%m_%Y_%H_%M_%S")
name_mensj_chatbot = prefix + "_mensajes_chatbot.xlsx"

In [ ]:
# Cargar variables de entorno desde el archivo .env
load_dotenv()

# Obtener variables de entorno
POSTGRES_HOST = os.getenv("POSTGRES_HOST")
POSTGRES_USER = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_PORT = os.getenv("POSTGRES_PORT")

# OpenAI
api_key = os.getenv("OPENAI_API_KEY") or os.getenv("OPEN_API_KEY")

In [7]:
# Conexión S3
s3 = boto3.client(
    's3',
    endpoint_url=os.environ["MLFLOW_S3_ENDPOINT_URL"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"], 
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"]
)

In [8]:
# 1. Conexión a la base de datos
conexion = psycopg2.connect(
    host=POSTGRES_HOST,
    database="saacdata",
    user=POSTGRES_USER,
    password=POSTGRES_PASSWORD,
    port=POSTGRES_PORT
)

In [9]:
# 2. Creación del cursor
cursor = conexion.cursor()

In [10]:
# 3. Ejecución de una consulta
cursor.execute("SELECT version();")

In [11]:
# 4. Obtención de resultados
db_version = cursor.fetchone()
print(f"Versión de PostgreSQL conectada: {db_version[0]}")

Versión de PostgreSQL conectada: PostgreSQL 17.10 (Debian 17.10-1.pgdg12+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 12.2.0-14+deb12u1) 12.2.0, 64-bit


### Chat sesiones

In [12]:
query_chats_sesiones = """
SELECT
    m.*,
    s.usuario_cedula,
    s.alias_sesion,
    s.model_used,
    s.regimenes
FROM "TH".chat_mensajes m
INNER JOIN "TH".chat_sesiones s
    ON m.session_id = s.session_id
WHERE s.usuario_cedula <> '0922663208'
  AND m.role <> 'tool';
"""

In [13]:
df_sesiones_mensajes = pd.read_sql_query(query_chats_sesiones, conexion)

/tmp/ipykernel_2113/645492952.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sesiones_mensajes = pd.read_sql_query(query_chats_sesiones, conexion)


In [14]:
df_sesiones_mensajes.shape

(1419, 12)

In [15]:
df_sesiones_mensajes.keys()

Index(['id', 'session_id', 'content', 'role', 'tool_calls', 'tokens_used',
       'latency_ms', 'created_at', 'usuario_cedula', 'alias_sesion',
       'model_used', 'regimenes'],
      dtype='object')

In [16]:
df_sesiones_mensajes["usuario_cedula"] = df_sesiones_mensajes["usuario_cedula"].str.strip()

In [17]:
# df_completo_column[df_completo_column["usuario_cedula"]].unique()
list_cedulas = df_sesiones_mensajes["usuario_cedula"].unique()
print("Usuarios que han escrito mensajes:", len(list_cedulas))

Usuarios que han escrito mensajes: 199


#### Get datos usuarios

In [18]:
query_get_usuarios_selected = f"""SELECT * FROM "TH".chat_usuarios where cedula in ('{"', '".join(map(str, list_cedulas))}')"""

In [19]:
df_usuarios = pd.read_sql_query(query_get_usuarios_selected, conexion)

/tmp/ipykernel_2113/2315406093.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_usuarios = pd.read_sql_query(query_get_usuarios_selected, conexion)


In [20]:
"Cantidad de usuarios", df_usuarios.shape # 188 el 20/07/2026

('Cantidad de usuarios', (199, 10))

In [21]:
# Cerrar cursor
cursor.close()

# 5. Cierre de la conexión
if conexion:
    conexion.close()

#### Asociacion de mensaje humano respuesta

In [22]:
df_conversaciones = (
    df_sesiones_mensajes
    .sort_values("created_at")
    .reset_index()
    .rename(columns={"index": "indice_original"})
    .copy()
)

In [23]:
df_conversaciones = df_conversaciones[df_conversaciones["role"].isin(["human", "ai"])].copy()

In [24]:
# Agrupar el mensaje por su sesion
df_conversaciones["siguiente_role"] = (
    df_conversaciones
    .groupby("session_id")["role"]
    .shift(-1)
)

df_conversaciones["siguiente_content"] = (
    df_conversaciones
    .groupby("session_id")["content"]
    .shift(-1)
)

In [25]:
df_mensajes_human = df_conversaciones[df_conversaciones["role"] == "human"].copy()

In [26]:
df_mensajes_human["respuesta_ia"] = (
    df_mensajes_human["siguiente_content"]
    .where(df_mensajes_human["siguiente_role"] == "ai")
)

In [27]:
df_mensajes_human.head(2)

,indice_original,id,session_id,content,role,tool_calls,tokens_used,latency_ms,created_at,usuario_cedula,alias_sesion,model_used,regimenes,siguiente_role,siguiente_content,respuesta_ia
0,5,14,137c58f3-d7c3-4c98-8b77-33413d58a834,LLAMAME mICKY,human,None,None,NaN,2026-06-29 19:09:11.088520+00:00,0930871157,M1k3,gpt-5.4-mini or gemini-3-flash-preview,[0],ai,Desde ahora te llamaré mICKY,Desde ahora te llamaré mICKY
2,7,16,a9e131a1-df72-4d8f-9292-d641b4a3fd16,llamame Mike,human,None,None,NaN,2026-06-29 19:09:32.203472+00:00,0930871157,mICKY,gpt-5.4-mini or gemini-3-flash-preview,[0],ai,Desde ahora te llamaré Mike,Desde ahora te llamaré Mike


In [28]:
df_mensajes_human.shape, df_mensajes_human[df_mensajes_human["siguiente_content"] == df_mensajes_human["respuesta_ia"]].shape, df_mensajes_human.shape[0]-df_mensajes_human[df_mensajes_human["siguiente_content"] == df_mensajes_human["respuesta_ia"]].shape[0]

((710, 16), (707, 16), 3)

In [29]:
df_mensajes_human[df_mensajes_human["siguiente_content"] != df_mensajes_human["respuesta_ia"]]

,indice_original,id,session_id,content,role,tool_calls,tokens_used,latency_ms,created_at,usuario_cedula,alias_sesion,model_used,regimenes,siguiente_role,siguiente_content,respuesta_ia
84,100,186,52e99c0a-faf3-44f9-9512-124af12ed413,Sí por favor.,human,None,None,NaN,2026-06-30 19:09:07.879206+00:00,1104861149,,gpt-5.4-mini or gemini-3-flash-preview,[8],human,"Sí, por favor.",NaN
196,256,335,fa217c7d-10ee-4fd7-9b0a-72864b4fafb6,cuales son los requisitos del puesto?''''''''''??\n,human,None,None,NaN,2026-07-01 13:48:58.086533+00:00,0923818991,,gpt-5.4-mini or gemini-3-flash-preview,[0],human,"pero, oregunte sobre los reusiitqtos del puesto?",NaN
672,787,985,b1e80d0a-437b-4b57-9deb-ca5a429806dd,hola se me olvido poner a mi revisor de informe de actividades,human,None,None,NaN,2026-07-03 02:51:00.950990+00:00,0930519921,,gpt-5.4-mini or gemini-3-flash-preview,[0],NaN,NaN,NaN


In [30]:
df_mensajes_human[df_mensajes_human["respuesta_ia"].isnull()]

,indice_original,id,session_id,content,role,tool_calls,tokens_used,latency_ms,created_at,usuario_cedula,alias_sesion,model_used,regimenes,siguiente_role,siguiente_content,respuesta_ia
84,100,186,52e99c0a-faf3-44f9-9512-124af12ed413,Sí por favor.,human,None,None,NaN,2026-06-30 19:09:07.879206+00:00,1104861149,,gpt-5.4-mini or gemini-3-flash-preview,[8],human,"Sí, por favor.",NaN
196,256,335,fa217c7d-10ee-4fd7-9b0a-72864b4fafb6,cuales son los requisitos del puesto?''''''''''??\n,human,None,None,NaN,2026-07-01 13:48:58.086533+00:00,0923818991,,gpt-5.4-mini or gemini-3-flash-preview,[0],human,"pero, oregunte sobre los reusiitqtos del puesto?",NaN
672,787,985,b1e80d0a-437b-4b57-9deb-ca5a429806dd,hola se me olvido poner a mi revisor de informe de actividades,human,None,None,NaN,2026-07-03 02:51:00.950990+00:00,0930519921,,gpt-5.4-mini or gemini-3-flash-preview,[0],NaN,NaN,NaN


In [31]:
df_mensajes_human[~(df_mensajes_human["tool_calls"].isnull())]

,indice_original,id,session_id,content,role,tool_calls,tokens_used,latency_ms,created_at,usuario_cedula,alias_sesion,model_used,regimenes,siguiente_role,siguiente_content,respuesta_ia


### PREPROCESSING

In [32]:
s3.download_file('processed-data', 'th/dataframe_completo.csv', 'mensajes_al_chatbot_tmp.csv')
df_history_processed = pd.read_csv('mensajes_al_chatbot_tmp.csv')
# remove the temporary file
os.remove('mensajes_al_chatbot_tmp.csv')

In [33]:
df_mensajes_human = df_mensajes_human[~df_mensajes_human['id'].isin(df_history_processed['id'])].copy()

In [34]:
print("Cantidad de nuevos mensajes", df_mensajes_human.shape)

Cantidad de nuevos mensajes (0, 16)


In [37]:
if df_mensajes_human.empty:
    raise SystemExit("No hay mensajes para procesar. Ejecución detenida.")

SystemExit: No hay mensajes para procesar. Ejecución detenida.

/usr/local/lib/python3.10/dist-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### Analisis de sentimiento

In [ ]:
model_name = "tabularisai/multilingual-sentiment-analysis"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

@misc{tabularisai2025multilingualsentiment,  
  author    = {Vadim Borisov and Samuel Gyamfi and Richard H. Schreiber},  
  title     = {Multilingual Sentiment Analysis},  
  year      = {2025},  
  doi       = {10.57967/hf/5968},  
  url       = {https://huggingface.co/tabularisai/multilingual-sentiment-analysis},  
  publisher = {Hugging Face},  
  note      = {Revision 69afb83}  
}

In [ ]:
def predict_sentiment(texts):
    inputs = tokenizer(texts, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
    sentiment_map = {0: "Very Negative", 1: "Negative", 2: "Neutral", 3: "Positive", 4: "Very Positive"}
    return [sentiment_map[p] for p in torch.argmax(probabilities, dim=-1).tolist()]

In [ ]:
texts = df_mensajes_human["content"].to_list()

In [ ]:
# for text, sentiment in zip(texts, predict_sentiment(texts)):
#     print(f"Text: {text}\nSentiment: {sentiment}\n")
# crear dataframe con resultado y el analisis de sentimiento
sentiments = predict_sentiment(texts)

In [ ]:
df_resultado_sentiment = df_mensajes_human.copy()
df_resultado_sentiment["sentiment"] = sentiments

In [ ]:
df_resultado_sentiment.shape

In [ ]:
df_resultado_sentiment.head(2)

In [ ]:
# Para grafico de sentimientos por cantidad de mensajes desde df_resultado_sentiment
orden_sentimientos = df_resultado_sentiment["sentiment"].unique()
sentiment_counts = (
    df_resultado_sentiment["sentiment"]
    .value_counts()
    .reindex(orden_sentimientos, fill_value=0)
)

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(sentiment_counts.index, sentiment_counts.values)
plt.xlabel("Sentimiento")
plt.ylabel("Cantidad de mensajes")
plt.title("Distribucion de sentimientos en los mensajes")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
df_resultado_sentiment[df_resultado_sentiment["sentiment"] == "Very Positive"][["content", "sentiment"]]

In [ ]:
df_resultado_sentiment[df_resultado_sentiment["sentiment"] == "Very Negative"][["content", "sentiment"]]

In [ ]:
df_resultado_sentiment.head(2) # analisis

In [ ]:
# df_resultado_sentiment.to_csv("data.csv", index=False)
# s3.upload_file('data.csv', 'processed-data', 'th/data.csv')

### Analisis con GPT

#### tool:
1. "Tool - get novedad": Recibe el token (de la metadata). Retorna: { estadoActual, fechaLimite, tieneNovedad }. **ATENCIÓN: Solo devuelve información del usuario actual. No utilizar para consultar sobre otras personas.**
2. "Tool - get subrogantes": No recibe parámetros. Retorna una lista completa de personas subrogadas, su cargo, unidad y quién los subroga. **(Información pública)**.
3. "Tool - get antiguedad": Recibe la identificación del usuario. Retorna la cantidad de años laborados. **ATENCIÓN: Solo devuelve información del usuario actual.**
4. "Tool - get subrogante cargo": Recibe la unidad de interés (nombre completo o siglas) y/o el cargo (ejemplo: decano, subdecano, director, gerente). En caso de que el usuario no envíe el cargo, pasar el parámetro como vacío (""). Retorna quién y hasta cuándo la están subrogando. **(Información pública)**.
5. "Tool - agente normativa": Herramienta de tipo AI Agent. Recibe la consulta íntegra del usuario sobre leyes, reglamentos, procesos internos o estatutos. Retorna una respuesta analizada, estructurada y oficial. USO OBLIGATORIO para cualquier duda conceptual de Talento Humano, normativas institucionales o pasos a realizar que no necesariamente piden al modelo que lo haga. **PROHIBIDO usar para preguntas de tipo "quién es" o "quién está a cargo de".**
6. "Tool - generate cv": Recibe el parámetro `format` (cuyo valor debe ser obligatoriamente "A" o "B"). Genera el CV del usuario actual en el diseño elegido. Retorna un objeto JSON con: { message, fileName, downloadUrl }.
7. "Tool - get autoridad": Recibe los parámetros `unidad` and `cargo`. Si el usuario solo menciona el cargo, asigna el mismo valor a ambos parámetros. Si menciona frases como "quien esta de decano de postgrados", identifica la unidad ("postgrados") y el cargo ("decano"). Ejemplos: "quien es la rectora" -> unidad: "rectora", cargo: "rectora"; "quien esta de decano de postgrados" -> unidad: "postgrados", cargo: "decano". Retorna un mensaje con el nombre de la persona, cargo, unidad, fechas de gestión y si está siendo subrogado o no. **(Información pública)**.
8. "Tool - get autoridad by name": Recibe el parámetro `nombre` (nombre de la persona por la que se consulta). Busca en el registro institucional y retorna su cargo, unidad, periodo de gestión y estado de subrogación. **(Información pública)**.
9. "Tool - update document": Recibe de forma exclusiva el parámetro cedula o votacion. REGLA CRÍTICA: Tienes prohibido ejecutar esta tool sin confirmar qué tipo de documento es. Esta tool devolverá un objeto JSON con el resultado o error. ESTÁ ESTRICTAMENTE PROHIBIDO mostrar llaves {}, código o formato JSON al usuario. Debes leer el JSON internamente y comunicar el resultado en lenguaje natural y empático.
10. "Tool - update alias": Recibe el parámetro `alias`. Llama a esta tool ÚNICAMENTE cuando el usuario ordene explícitamente cambiar su nombre de trato (ej. "llámame X"). **ESTÁ ESTRICTAMENTE PROHIBIDO ejecutar esta tool si el usuario solo está preguntando cuál es su nombre o cómo le dices.**


#### IA

In [ ]:
if not api_key:
    raise ValueError(
        "No se encontró OPENAI_API_KEY ni OPEN_API_KEY en las variables de entorno."
    )

client = OpenAI(api_key=api_key)

In [ ]:
PROMPT_CLASIFICAR_TOOL_BATCH = """
You are a strict intent router for the ESPOL Human Talent chatbot.

Task:
Analyze each Spanish user message independently and determine which available
tools can resolve the user's explicit or clearly actionable requests.

The input will be a JSON array with this structure:

[
    {
        "id": 0,
        "mensaje": "user message"
    }
]

You must analyze every message independently.

Output constraints:

1. Output only one valid RAW JSON array.
2. Return exactly one result for every input message.
3. Preserve the same "id" received in the input.
4. Preserve the same order as the input.
5. Each result must have exactly these properties:

   * "id": integer
   * "usa_tool": boolean
   * "tools": array

6. Set "usa_tool" to true when at least one available tool can resolve
   at least one explicit or clearly actionable user request.

7. Set "usa_tool" to false when none of the available tools can resolve
   the message.

8. When "usa_tool" is false, "tools" must be an empty array.

9. When "usa_tool" is true, "tools" must contain one or more allowed tool names.

10. Include only real tool names listed in "Allowed tools".

11. Do not repeat the same tool more than once.

12. Preserve the order in which the user's requests appear.

13. Analyze each message independently.
    Never use information from one message to classify another message.

14. Do not combine results from different messages.

Intent rules:

15. Select a tool only when the user is currently asking the chatbot to perform,
retrieve, verify, explain, generate, update, or consult something supported
by that tool.

16. Do not select a tool merely because the message contains a keyword
related to a tool.

17. A single isolated word, noun, topic, position name, document name,
or person name is not sufficient by itself to trigger a tool when
the user's intention is unclear.

18. Short messages can trigger a tool only when they express a clear
question, command, or actionable request.

19. If the message is ambiguous and does not contain a clear request
or question, return usa_tool=false and tools=[].

20. Do not infer unsupported intentions from vague or incomplete messages.

21. Past statements, examples, quoted text, hypothetical scenarios,
or descriptions of previous requests must not trigger a tool unless
they also contain a current supported request.

22. When multiple supported requests appear in the same message,
include all corresponding tools once, preserving request order.

Allowed tools:

* Tool - get novedad
* Tool - get subrogantes
* Tool - get antiguedad
* Tool - get subrogante cargo
* Tool - agente normativa
* Tool - generate cv
* Tool - get autoridad
* Tool - get autoridad by name
* Tool - update document
* Tool - update alias

Tool definitions:

* Tool - get novedad
  Use when the current user explicitly asks about their own pending novelty,
  novelty status, deadline, or whether they currently have a novelty.

* Tool - get subrogantes
  Use when the user requests the general or complete list of current
  substitutions or subrogations.

* Tool - get antiguedad
  Use when the current user asks about their own employment seniority,
  years worked, length of service, or institutional tenure.

* Tool - get subrogante cargo
  Use when the user asks who is temporarily substituting or subrogating
  a specific position, role, or unit.

* Tool - agente normativa
  Use when the user asks a question about Human Talent policies, laws,
  institutional rules, requirements, procedures, benefits, permissions,
  vacations, licenses, or steps to follow.

* Tool - generate cv
  Use when the current user explicitly asks to generate, create, produce,
  prepare, or download their CV.

* Tool - get autoridad
  Use when the user asks who currently holds a specific position
  or authority role.

* Tool - get autoridad by name
  Use when the user provides a person's name and asks about that person's
  position, unit, authority information, or substitution status.

* Tool - update document
  Use when the current user explicitly asks to update their ID card
  document or voting certificate.

* Tool - update alias
  Use when the user explicitly asks to change the name or alias used
  to address them.

Examples:

Input:
[
    {"id": 0, "mensaje": "hola buenos días"},
    {"id": 1, "mensaje": "dime mi antigüedad laboral"},
    {"id": 2, "mensaje": "quién es el rector"},
    {"id": 3, "mensaje": "vacaciones"}
]

Output:
[
    {"id": 0, "usa_tool": false, "tools": []},
    {"id": 1, "usa_tool": true, "tools": ["Tool - get antiguedad"]},
    {"id": 2, "usa_tool": true, "tools": ["Tool - get autoridad"]},
    {"id": 3, "usa_tool": false, "tools": []}
]
"""

In [ ]:
TOOLS_PERMITIDAS = {
    "Tool - get novedad",
    "Tool - get subrogantes",
    "Tool - get antiguedad",
    "Tool - get subrogante cargo",
    "Tool - agente normativa",
    "Tool - generate cv",
    "Tool - get autoridad",
    "Tool - get autoridad by name",
    "Tool - update document",
    "Tool - update alias",
}

#### CLASIFICA MENSAJE(S) POR TOOLS

In [ ]:
def clasificar_tools_batch(
    mensajes: list[str]
) -> list[dict[str, Any]]:

    if not mensajes:
        return []

    input_data = [
        {
            "id": i,
            "mensaje": mensaje
        }
        for i, mensaje in enumerate(mensajes)
    ]

    response = client.responses.create(
        model="gpt-5.4-nano",
        instructions=PROMPT_CLASIFICAR_TOOL_BATCH,
        input=json.dumps(
            input_data,
            ensure_ascii=False
        ),
        temperature=0,
        max_output_tokens=max(500, len(mensajes) * 50),
    )

    texto_respuesta = response.output_text.strip()

    try:
        resultados = json.loads(texto_respuesta)

    except json.JSONDecodeError as error:
        raise ValueError(
            f"El modelo no devolvió JSON válido: "
            f"{texto_respuesta!r}"
        ) from error

    if not isinstance(resultados, list):
        raise ValueError(
            "La respuesta debe ser una lista JSON."
        )

    if len(resultados) != len(mensajes):
        raise ValueError(
            f"Se enviaron {len(mensajes)} mensajes "
            f"pero se recibieron {len(resultados)} resultados."
        )

    resultados_validados = []

    for esperado_id, resultado in enumerate(resultados):

        if not isinstance(resultado, dict):
            raise ValueError(
                f"El resultado {esperado_id} no es un objeto."
            )

        if set(resultado.keys()) != {
            "id",
            "usa_tool",
            "tools"
        }:
            raise ValueError(
                f"Estructura incorrecta en resultado "
                f"{esperado_id}: {resultado}"
            )

        id_resultado = resultado["id"]
        usa_tool = resultado["usa_tool"]
        tools = resultado["tools"]

        if id_resultado != esperado_id:
            raise ValueError(
                f"ID inesperado. Esperado: {esperado_id}, "
                f"recibido: {id_resultado}"
            )

        if not isinstance(usa_tool, bool):
            raise ValueError(
                '"usa_tool" debe ser booleano.'
            )

        if not isinstance(tools, list):
            raise ValueError(
                '"tools" debe ser una lista.'
            )

        tools = list(dict.fromkeys(tools))

        tools_invalidas = [
            tool
            for tool in tools
            if tool not in TOOLS_PERMITIDAS
        ]

        if tools_invalidas:
            raise ValueError(
                f"Tools no permitidas: {tools_invalidas}"
            )

        if usa_tool != bool(tools):
            raise ValueError(
                f"Inconsistencia en mensaje {esperado_id}: "
                f"usa_tool={usa_tool}, tools={tools}"
            )

        resultados_validados.append({
            "usa_tool": usa_tool,
            "tools": tools
        })

    return resultados_validados

#### PREPARA df_mensajes_prueba PARA CLASIFICAR

In [ ]:
df_mensajes_prueba = (
    df_resultado_sentiment[["content"]]
    .copy()
)

# Selecciona únicamente la columna content del DataFrame.
# nulos en vacio
# convertir valores a texto.
# Elimina espacios al inicio y al final de cada mensaje.
df_mensajes_prueba["content"] = (
    df_mensajes_prueba["content"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# elimina mensajes vacíos y serie de Pandas en una lista normal de Python
df_mensajes_prueba = (
    df_mensajes_prueba[
        df_mensajes_prueba["content"] != ""
    ]
    .reset_index()
    .rename(columns={"index": "indice_original"})
)

In [ ]:
df_mensajes_prueba.shape, df_resultado_sentiment.shape, "Deberia ser igual", df_mensajes_prueba.shape[0] == df_resultado_sentiment.shape[0]

In [ ]:
df_mensajes_prueba.keys(), df_resultado_sentiment.keys()

#### PROCESAR TODOS LOS MENSAJES

In [ ]:
def procesar_todos_los_mensajes(
    df_mensajes_prueba,
    batch_size=20
):

    # Copia para no modificar el dataframe original
    df = df_mensajes_prueba.copy()

    total_original = len(df)

    print(
        f"Total de filas originales: "
        f"{total_original}"
    )

    # --------------------------------------------------
    # 1. Obtener mensajes únicos
    # --------------------------------------------------

    # mensajes_unicos = (
    #     df["content"]
    #     .dropna()
    #     .astype(str)
    #     .drop_duplicates()
    #     .tolist()
    # )
    df["_content_key"] = (
        df["content"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )
    
    df_unicos = df.drop_duplicates(
        subset="_content_key",
        keep="first"
    )
    
    mensajes_unicos = (
        df_unicos["content"]
        .tolist()
    )
    

    total_unicos = len(mensajes_unicos)

    print(
        f"Mensajes únicos a procesar: "
        f"{total_unicos}"
    )

    print(
        f"Mensajes duplicados evitados: "
        f"{total_original - total_unicos}"
    )

    # --------------------------------------------------
    # 2. Diccionario:
    #    mensaje -> clasificación
    # --------------------------------------------------

    cache_resultados = {}

    # --------------------------------------------------
    # 3. Procesar en batches
    # --------------------------------------------------

    for inicio in range(
        0,
        total_unicos,
        batch_size
    ):

        fin = min(
            inicio + batch_size,
            total_unicos
        )

        batch = mensajes_unicos[inicio:fin]

        print(
            f"Procesando mensajes "
            f"{inicio + 1}-{fin} "
            f"de {total_unicos}"
        )

        try:

            resultados = clasificar_tools_batch(
                batch
            )

            for mensaje, resultado in zip(
                batch,
                resultados
            ):
                cache_resultados[mensaje] = {
                    "usa_tool":
                        resultado["usa_tool"],
                    "tools":
                        resultado["tools"],
                    "error":
                        None
                }

        except Exception as error:

            # Si falla todo el batch,
            # registramos el error
            for mensaje in batch:
                cache_resultados[mensaje] = {
                    "usa_tool": None,
                    "tools": [],
                    "error": str(error)
                }

    # --------------------------------------------------
    # 4. Mapear resultados al dataframe ORIGINAL
    # --------------------------------------------------

    df["usa_tool"] = df["content"].map(
        lambda x:
        cache_resultados.get(
            str(x),
            {}
        ).get(
            "usa_tool"
        )
        if pd.notna(x)
        else False
    )

    df["tools"] = df["content"].map(
        lambda x:
        cache_resultados.get(
            str(x),
            {}
        ).get(
            "tools",
            []
        )
        if pd.notna(x)
        else []
    )

    df["error"] = df["content"].map(
        lambda x:
        cache_resultados.get(
            str(x),
            {}
        ).get(
            "error"
        )
        if pd.notna(x)
        else None
    )

    print(
        "Procesamiento completado."
    )

    return df

In [ ]:
df_resultados_completos = (
    procesar_todos_los_mensajes(
        df_mensajes_prueba,
        batch_size=20
    )
)

In [ ]:
df_resultado_sentiment_cp = df_resultado_sentiment.merge(
    df_resultados_completos,
    left_index=True,
    right_on="indice_original",
    how="left"
)

In [ ]:
df_resultado_sentiment_cp.keys()

In [ ]:
df_resultado_sentiment = df_resultado_sentiment_cp.drop(["indice_original", "indice_original_y", "content_y"], axis=1).rename(columns={"indice_original_x": "indice_original", "content_x": "content"}).copy()

In [ ]:
df_resultado_sentiment.shape

In [ ]:
df_resultado_sentiment.keys()

### Guardar DF del chat completo

In [ ]:
df_history_processed_concated = pd.concat(
    [df_history_processed, df_resultado_sentiment], 
    ignore_index=True
)

In [ ]:
df_history_processed.shape, df_resultado_sentiment.shape, df_history_processed_concated.shape

In [ ]:
df_history_processed.shape[0] + df_resultado_sentiment.shape[0] == df_history_processed_concated.shape[0]

In [ ]:
df_history_processed_concated.to_csv("dataframe_completo.csv", index=False)
s3.upload_file('dataframe_completo.csv', 'processed-data', 'th/dataframe_completo.csv')
os.remove("dataframe_completo.csv")

### POSTPROCESSING

In [ ]:
# Filtro de mensajes
list_mensajes_no = ["llamame", "LLAMAME", "llámame", "Llámame por favor", "llámame por favor", "dime asi", "me puedes llamar", "cuenta hasta"]

In [ ]:
list_complete_mensaje_no = ["que puedo hacer", "Sí por favor.", "Hola", "Sí, por favor.", "si por favor", "listo ayudame", "2+2?", 
                            "ERROR [23502] [IBM][DB2/AIX64] SQL0407N Assignment of a NULL value to a NOT NULL column \"TBSPACEID=5, TABLEID=1233, COLNO=1\" is not allowed.",
                            "ok", "si", "sí", "listo", "ok, gracias", "ok gracias", "ok gracias.", "ok, gracias.", "ok gracias!", 
                            "ok, gracias!", "ok gracias!!", "ok, gracias!!", "ok gracias!!!", "ok, gracias!!!", "gracias", "a", 
                            "hello", "hazlo", "si hazlo", "sí por favor", "ahora me vas a decir papi y tu eres mi mujer", "Dime Churris"]
list_complete_mensaje_no_lower = [msg.lower() for msg in list_complete_mensaje_no]

In [ ]:
# cada columne tiene un mensaje de human
df_tmp = df_resultado_sentiment[(~df_resultado_sentiment["content"].str.contains("|".join(list_mensajes_no), case=False, na=False))].copy()

# Que el contenido completo no coincida con ninguno de los mensajes de exclusión
filtro_mensaje_no = (~df_tmp["content"].str.strip().str.lower().isin(list_complete_mensaje_no_lower))

df_tmp_res = df_tmp[filtro_mensaje_no]

In [ ]:
df_tmp_res.shape

In [ ]:
df_tmp_res.keys()

In [ ]:
df_res_to_th = df_tmp_res[["content", 'respuesta_ia']].copy()

In [ ]:
df_res_to_th.shape

In [ ]:
df_res_to_th.keys()

In [ ]:
# # que sea humano y contenga mensajes del dataframe
# df_new_content = df_mensajes_human[(df_mensajes_human["role"] == "human") & (~df_mensajes_human["content"].str.contains("|".join(list_mensajes_no), case=False))]["content"].copy()
# df_new_content

In [ ]:
# # que no contenga ninguna de las palabras en list_mensajes_no
# filtro_mensaje_no = ~df_new_content.str.strip().str.lower().isin(list_complete_mensaje_no_lower)

# res_msg_user_filter = df_new_content[filtro_mensaje_no]
# res_msg_user_filter

### Guardar mensajes del chat

In [ ]:
df_res_to_th.to_excel(name_mensj_chatbot, index=False)
s3.upload_file(name_mensj_chatbot, 'processed-data', 'th/mensajes/' + name_mensj_chatbot)
os.remove(name_mensj_chatbot)